In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers datasets accelerate

In [3]:
import os, json
import numpy as np, pandas as pd, torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding)
from datasets import Dataset

In [4]:
#identical to XGboost

BIGO_DIR = '/content/drive/MyDrive/BIGO'
rows = []
with open(os.path.join(BIGO_DIR, 'java_data.jsonl'), encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
df = pd.DataFrame(rows)
print('Shape:', df.shape)

Shape: (4900, 5)


In [5]:
le = LabelEncoder()
y = le.fit_transform(df['complexity'])
groups = df['problem'].values
idx = np.arange(len(df))
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(idx, y, groups))
print('train:', len(tr), 'test:', len(te))
assert set(groups[tr]).isdisjoint(set(groups[te])), 'LEAK!'

train: 3768 test: 1132


In [6]:
MODEL = 'microsoft/graphcodebert-base'
tok = AutoTokenizer.from_pretrained(MODEL)

def make_ds(indices):
    texts  = [df['src'].iloc[i] for i in indices]
    labels = [int(y[i]) for i in indices]
    enc = tok(texts, truncation=True, max_length=512)   # long code cut at 512 tokens
    enc['labels'] = labels
    return Dataset.from_dict(enc)

train_ds = make_ds(tr)
test_ds  = make_ds(te)
print(train_ds)

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3768
})


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(le.classes_))

args = TrainingArguments(
    output_dir='/content/out',
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    save_strategy='no',
    report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=DataCollatorWithPadding(tok))
trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/graphcodebert-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.decoder.weight     | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.

Step,Training Loss
50,1.956780
100,1.899089
150,1.782499
200,1.500865
250,1.181629
300,1.200847
350,0.964228
400,0.935005
450,0.852755
500,0.756414


TrainOutput(global_step=1884, training_loss=0.6455665490176774, metrics={'train_runtime': 487.6587, 'train_samples_per_second': 30.907, 'train_steps_per_second': 3.863, 'total_flos': 3965787854438400.0, 'train_loss': 0.6455665490176774, 'epoch': 4.0})

In [8]:
pred = trainer.predict(test_ds)
y_pred = pred.predictions.argmax(-1)
y_true = np.array(test_ds['labels'])
acc = accuracy_score(y_true, y_pred)
print(f'TRANSFORMER (GraphCodeBERT) | problem-split accuracy = {acc:.4f}')
print(classification_report(y_true, y_pred, target_names=le.classes_))

TRANSFORMER (GraphCodeBERT) | problem-split accuracy = 0.5442
              precision    recall  f1-score   support

    constant       0.78      0.95      0.86       214
       cubic       0.18      0.03      0.05       297
      linear       0.78      0.45      0.57       249
        logn       0.86      0.83      0.84       202
       nlogn       0.73      0.77      0.75       137
          np       0.02      1.00      0.04         6
   quadratic       0.18      0.52      0.27        27

    accuracy                           0.54      1132
   macro avg       0.50      0.65      0.48      1132
weighted avg       0.61      0.54      0.55      1132



In [9]:
out = os.path.join(BIGO_DIR, 'graphcodebert_bigo')
model.save_pretrained(out)
tok.save_pretrained(out)
print('saved to', out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/BIGO/graphcodebert_bigo
